In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 재샘플링과 Order Analysis — 비정상 회전 신호 분석 + TSA

## 학습 목표
- 회전 속도가 변하는 신호에서 시간 영역 FFT 가 퍼지는 현상과 각도 기반 재샘플링 (order tracking) 을 다룬다
- Zero-padding 과 trig_rot=1 두 가지로 leakage 를 해결한다
- TSA (Time Synchronous Averaging) 로 주기적 이상치를 드러낸다
- 하이패스 + 힐베르트 포락선 잔차 분석을 수행한다

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from utils import fft, filtering, filtering_zerophase, resampling, hilbert_envelope

---

## 실습 1. 신호 생성 — 회전속도가 시간에 따라 변하는 기어 신호

- 회전 속도 $f_{rot}(t) = 1 + t$ [Hz] (선형 증가)
- 누적 위상 $\phi(t) = \int_0^t f_{rot}(\tau)\,d\tau$
- 기어 이빨이 `n_gear = 3`개 → 진동 신호 $v(t) = \sin(n_{gear}\cdot 2\pi\phi(t))$

In [ ]:
fs = 1000
T = 20
t = np.arange(1/fs, T + 1/fs, 1/fs)
n_gear = 3

f_rotation = 1 + t                       # 순시 회전 주파수 (Hz)
phase_deg = np.cumsum(f_rotation) * 360 / fs
phase_rad = 2 * np.pi * phase_deg / 360
v = np.sin(n_gear * phase_rad)

In [ ]:
# signal overview 3x1 — rotation speed + waveform + time-domain FFT
fig, axes = plt.subplots(3, 1, figsize=(11, 7))

axes[0].plot(t, f_rotation, color='C0', lw=1.0)
axes[0].set_ylabel('Rotation speed (rev/s)')
axes[0].set_title('(a) Rotation speed vs time')

axes[1].plot(t, v, color='C1', lw=0.6)
axes[1].set_ylabel('Amplitude')
axes[1].set_title('(b) Vibration waveform — chirp-like due to speed change')
axes[1].set_xlim(0, min(t[-1], 5))

f_time, A_time = fft(v, fs)
axes[2].plot(f_time, A_time, color='C3', lw=0.8)
axes[2].set_xlim(0, 50); axes[2].set_ylabel('Amplitude')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_title('(c) Time-domain FFT — rotation components spread')

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5))
axes[0].plot(t, v, color='C0')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('v')
axes[0].set_title('Total Signal')
axes[1].plot(t, v, '.-', color='C0', markersize=3)
axes[1].set_xlim([0, 5])
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('v')
axes[1].set_title('First 5s zoom — accelerating vibration')
fig.tight_layout(); plt.show()

---

## 실습 2. 시간 영역 FFT — 스펙트럼이 퍼지는 현상

회전 속도가 시간에 따라 변하면, 같은 기어 주파수 성분이 여러 주파수에 걸쳐 퍼져 나타난다.

In [ ]:
f, A = fft(v, fs)

plt.figure(figsize=(9, 4))
plt.plot(f, A, color='C0')
plt.xlim([0, 100])
plt.xlabel('Frequency (Hz)'); plt.ylabel('|Y|')
plt.title('Time Domain FFT — rotation components spread')
plt.tight_layout(); plt.show()

---

## 실습 3. 각도 기반 재샘플링 — Resampling Frequency 결정

- 전체 회전수 $N_{rot} = \phi_{\text{end}} / 360$
- 회전 1바퀴당 샘플 수 `fs_re` 를 결정 (데이터 기반 추정치를 참고, 여기서는 100으로 지정)

In [ ]:
N_rot = phase_deg[-1] / 360
N_samples = len(v)
fs_re_est = np.floor(N_samples / N_rot)

print(f'Total rotations              : {N_rot:.2f}')
print(f'Total samples                 : {N_samples}')
print(f'Samples per rotation (est.)   : {fs_re_est:.0f}')

fs_re = 100  # 회전 1바퀴당 100 샘플로 재샘플링

---

## 실습 4. 재샘플링 직접 구현 — 원리 이해

1. 각도 축에 등간격 격자 생성 (`degree_re_delta = 360 / fs_re`)
2. 각도-시간 보간: 등간격 각도에 해당하는 시간 `t_re`
3. 시간-신호 보간: `t_re` 시점의 신호 값 `v_re`

In [ ]:
starting = phase_deg[0]
ending = phase_deg[-1]
degree_re_delta = 360 / fs_re
degree_re = np.arange(starting + degree_re_delta, ending, degree_re_delta)

fx_t_re = interp1d(phase_deg, t)
t_re = fx_t_re(degree_re)

fx_v_re = interp1d(t, v)
v_re = fx_v_re(t_re)

print(f'Number of resampling points: {len(v_re)}')
print(f'First 4 angles    : {degree_re[:4]}')
print(f'Last 4 angles    : {degree_re[-4:]}')

---

## 실습 5. Order Analysis — 각도 축 FFT

재샘플링된 신호를 FFT 하면 주파수 축이 **Order** (회전당 주기 수) 축으로 바뀐다. 기어 이빨 수 `n_gear = 3` 이므로 **Order = 3** 에 첨예한 피크가 나타나야 한다.

In [ ]:
order, order_A = fft(v_re, fs_re)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(order, order_A, color='C0')
axes[0].set_xlabel('Order'); axes[0].set_ylabel('|Y|')
axes[0].set_title('Total Order Spectrum')
axes[1].plot(order, order_A, '-o', color='C0', markersize=4)
axes[1].set_xlim([2.8, 3.2])
axes[1].set_xlabel('Order'); axes[1].set_ylabel('|Y|')
axes[1].set_title('Order=3 zoom — still spread')
fig.tight_layout(); plt.show()

---

## 실습 6. Trig=0 재샘플링 (utils 함수 활용)

`utils.resampling` 의 `trig_rot=0` 옵션은 정수 회전수를 맞추지 않고 그대로 반환한다. → 신호 길이가 회전수의 정수배가 아니라서 주파수 해상도에 소수점 오차가 생기고, Order=3 근처 스펙트럼이 퍼진다.

In [ ]:
trig = 0
t_re, v_re, degree_re, fs_re = resampling(t, v, phase_deg, fs_re, trig)
N_re = len(v_re)
print(f'Resampled signal length: {N_re}, fs_re={fs_re}')

order, order_A = fft(v_re, fs_re)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(order, order_A, color='C0')
axes[0].set_xlabel('Order'); axes[0].set_ylabel('|Y|')
axes[0].set_title('Trig=0 Total Order Spectrum (spread)')
axes[1].plot(order, order_A, '-o', color='C0', markersize=4)
axes[1].set_xlim([2.95, 3.05])
axes[1].set_xlabel('Order'); axes[1].set_ylabel('|Y|')
axes[1].set_title('Trig=0: Order=3 peak spread')
fig.tight_layout(); plt.show()
order_trig0, A_trig0 = order.copy(), order_A.copy()

m = (order >= 2.9) & (order <= 3.1)
if order_A[m].size:
    concentration = 100 * np.sum(order_A[m]**2) / np.sum(order_A**2)
    print(f'Trig=0        — Order=3 concentration: {concentration:.1f} %')

**관찰**: Trig=0 은 정수 회전수를 맞추지 않아 Order=3 이 FFT bin 중심에서 벗어남 → spectral leakage 로 Order=3 근방이 퍼진다. 이를 해결하는 두 가지 방법을 다음 실습에서 본다.

---

## 실습 7. 해결방법 1 — Zero-padding

신호 뒤쪽을 0으로 채워 길이를 `fs_re`의 정수배로 맞춘다.

In [ ]:
pad_len = int(fs_re - np.mod(N_re, fs_re))
v_re_pad = np.append(v_re, np.zeros(pad_len))
print(f'Length after zero-padding: {len(v_re_pad)}')

order, order_A = fft(v_re_pad, fs_re)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(order, order_A, color='C0')
axes[0].set_xlabel('Order'); axes[0].set_ylabel('|Y|')
axes[0].set_title('After Zero-padding: Total Order Spectrum')
axes[1].plot(order, order_A, '-o', color='C0', markersize=4)
axes[1].set_xlim([2.95, 3.05])
axes[1].set_xlabel('Order'); axes[1].set_ylabel('|Y|')
axes[1].set_title('Zero-padding: Order=3 peak sharp')
fig.tight_layout(); plt.show()

# Order=3 근방 에너지 집중도
m = (order >= 2.9) & (order <= 3.1)
if order_A[m].size:
    concentration = 100 * np.sum(order_A[m]**2) / np.sum(order_A**2)
    print(f'Zero-padding  — Order=3 concentration: {concentration:.1f} %')
order_zp, A_zp = order.copy(), order_A.copy()

---

## 실습 8. 해결방법 2 — Trig=1 (정수 회전수만 유지)

`utils.resampling(..., trig_rot=1)` 은 정수 회전수를 넘는 뒤쪽 샘플을 잘라버린다. → 신호 길이가 정확히 n 바퀴에 해당한다.

In [ ]:
trig = 1
t_re, v_re, degree_re, fs_re = resampling(t, v, phase_deg, fs_re, trig)
N_re = len(v_re)
print(f'Trig=1 resampled length: {N_re}  ({N_re % fs_re == 0} → integer rotation multiple)')

order, order_A = fft(v_re, fs_re)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(order, order_A, color='C0')
axes[0].set_xlabel('Order'); axes[0].set_ylabel('|Y|')
axes[0].set_title('Trig=1 (integer rotations): Total Order Spectrum')
axes[1].plot(order, order_A, '-o', color='C0', markersize=4)
axes[1].set_xlim([2.95, 3.05])
axes[1].set_xlabel('Order'); axes[1].set_ylabel('|Y|')
axes[1].set_title('Trig=1: Order=3 peak sharp')
fig.tight_layout(); plt.show()

m = (order >= 2.9) & (order <= 3.1)
if order_A[m].size:
    concentration = 100 * np.sum(order_A[m]**2) / np.sum(order_A**2)
    print(f'Trig=1        — Order=3 concentration: {concentration:.1f} %')
order_trig1, A_trig1 = order.copy(), order_A.copy()

In [ ]:
# trig0/zeropad/trig1 comparison — 세 해결책을 Order=3 근방에서 나란히 비교
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
cases = [
    ('Trig=0',       order_trig0, A_trig0, 'C0'),
    ('Zero-padding', order_zp,    A_zp,    'C1'),
    ('Trig=1',       order_trig1, A_trig1, 'C2'),
]
for ax, (lbl, o, a, color) in zip(axes, cases):
    ax.plot(o, a, '-o', color=color, markersize=4, lw=1.0)
    ax.set_xlim(2.7, 3.3); ax.set_xlabel('Order')
    ax.set_title(lbl)
    ax.axvline(3.0, color='gray', ls=':', lw=0.8)
axes[0].set_ylabel('Amplitude')
fig.suptitle('Order=3 Zoom — Three Resampling Strategies')
plt.tight_layout(); plt.show()

**관찰**: Zero-padding 과 Trig=1 모두 Order=3 에 에너지가 집중된다. 두 방법 모두 leakage 를 근본적으로 억제. concentration % 수치로 정량 비교.

---

## 실습 9. 신호 생성 — 주기적 이상치 추가 (정상 vs 고장)

각 회전마다 동일한 위치(40번째 샘플)에 진폭 0.5 펄스를 더해 기어 1바퀴당 1회 이상치가 나타나는 고장 신호를 만든다.

In [ ]:
v_re_f = np.copy(v_re_pad)                # fault
v_re_n = np.copy(v_re_pad)                # normal
idx_fault = np.arange(40, len(v_re_f), fs_re)
v_re_f[idx_fault] = v_re_f[idx_fault] + 0.5

fig, axes = plt.subplots(2, 1, figsize=(9, 5))
axes[0].plot(v_re_f, color='C3', label='Fault')
axes[0].plot(v_re_n, color='C0', alpha=0.5, label='Normal')
axes[0].legend(); axes[0].set_title('Impact added — Total')
axes[1].plot(v_re_f, color='C3', label='Fault')
axes[1].plot(v_re_n, color='C0', alpha=0.5, label='Normal')
axes[1].set_xlim([0, 1000]); axes[1].legend()
axes[1].set_title('Zoom — periodic pulse (period=fs_re=100)')
fig.tight_layout(); plt.show()

---

## 실습 10. 노이즈 추가 — 이상치가 가려짐

In [ ]:
np.random.seed(1000)
v_re_f = v_re_f + np.random.randn(len(v_re_f)) / 2
v_re_n = v_re_n + np.random.randn(len(v_re_n)) / 2

plt.figure(figsize=(9, 4))
plt.plot(v_re_f, color='C3', label='Fault')
plt.plot(v_re_n, color='C0', alpha=0.5, label='Normal')
plt.xlim([0, 1000])
plt.xlabel('Samples'); plt.ylabel('v'); plt.legend()
plt.title('Impact buried in Noise — hard to distinguish')
plt.tight_layout(); plt.show()

---

## 실습 11. TSA — 신호를 회전주기로 잘라서 평균

- 5 회전(= `5 * fs_re` 샘플)씩 끊어 reshape
- 여러 세그먼트를 평균 → 주기 동기 성분(이상치)은 살아남고 비동기 노이즈는 상쇄

(TSA reshape는 `v_re` 길이가 `seg_len`의 정수 배일 때만 에러 없이 동작한다. 아래 안전장치로 여분 샘플을 절단하여 실습자가 파라미터를 바꿔도 안전하게 실행되도록 처리한다.)

In [ ]:
n_rot_TSA = 5
seg_len = n_rot_TSA * fs_re

# 안전장치: v_re 길이를 seg_len 배수로 맞춤
n_full = (len(v_re_f) // seg_len) * seg_len
v_re_f = v_re_f[:n_full]
v_re_n = v_re_n[:n_full]

v_reshape_f = v_re_f.reshape(seg_len, -1, order='F')
v_TSA_f = np.mean(v_reshape_f, axis=1)

v_reshape_n = v_re_n.reshape(seg_len, -1, order='F')
v_TSA_n = np.mean(v_reshape_n, axis=1)

print(f'segment shape: {v_reshape_f.shape} (seg_len={seg_len}, n_segments={v_reshape_f.shape[1]})')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6))
axes[0].plot(v_reshape_f, color='C0', alpha=0.15)
axes[0].plot(v_TSA_f, '--', color='C3', linewidth=2.5, label='TSA Mean')
axes[0].legend(); axes[0].set_title('Fault — Segments overlaid + Mean')
axes[1].plot(v_TSA_f, color='C3', label='Fault TSA')
axes[1].plot(v_TSA_n, color='C0', label='Normal TSA')
axes[1].set_xlabel('Samples (5 rotations)'); axes[1].legend()
axes[1].set_title('Fault vs Normal TSA — periodic Impact visible')
fig.tight_layout(); plt.show()

**관찰**: reshape 후 세그먼트 평균을 취하면 회전 **비동기** 성분(랜덤 노이즈, 다른 주파수)은 $1/\sqrt{N}$ 로 억제되고, **동기** 성분(회전과 같은 주기로 반복되는 임펄스·고조파)은 그대로 강화된다. Fault TSA 곡선에서 sharp peak 가 뚜렷이 드러난다.

---

## 실습 12. TSA 신호의 주파수 분석

주기적 이상치는 기어 오더(3) 와 그 배수 성분으로 나타난다.

In [ ]:
f_n, A_n = fft(v_TSA_n, fs_re)
f_f, A_f = fft(v_TSA_f, fs_re)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(f_f, A_f, color='C3', label='Fault')
axes[0].plot(f_n, A_n, color='C0', label='Normal')
axes[0].set_xlabel('Order'); axes[0].set_ylabel('|Y|'); axes[0].legend()
axes[0].set_title('TSA Spectrum (Total)')
axes[1].plot(f_f, A_f, color='C3', label='Fault')
axes[1].plot(f_n, A_n, color='C0', label='Normal')
axes[1].set_ylim([0, 0.05]); axes[1].legend()
axes[1].set_xlabel('Order'); axes[1].set_ylabel('|Y|')
axes[1].set_title('Zoom — Fault components (harmonics) sharp')
fig.tight_layout(); plt.show()

**관찰**: TSA 후 Order 축 FFT 에서는 기어 **Order=3 와 그 고조파** (6, 9, ...) 가 sharp 한 bin peak 으로 나타난다. 결함 신호에는 추가로 Order=3 의 **sideband** (예: 3±1/rot ratio) 나 저주파 impulse 성분이 보여 정상과 구분된다. 비동기 노이즈 성분은 평균 과정에서 억제되어 peak-to-floor 비가 크게 향상된다.

---

## 실습 13. 하이패스 필터링 후 잔차(residual) 분석

기본 기어 Order=3 성분을 제거하고 고주파 성분(이상치에 의한 고조파)만 남긴 뒤, 힐베르트 포락선으로 잔차 크기를 본다.

- `utils.bandpass` 로 10 ~ (fs_re/2 − 1) Order 통과 ⇒ 10 Order 이상을 통과시킨 것과 동일

(하이패스 10 Order 이상 대역을 통과시키므로 **기본 기어 Order=3과 그 배음(Order=6, 9)**까지 함께 제거된다. 잔차에는 결함 임펄스 + 고주파 공진 성분이 남는다.)

In [ ]:
f_low = 10.0
f_high = fs_re / 2 - 1.0
v_high_n = filtering_zerophase(v_TSA_n, fs_re, 'band', f_low=f_low, f_high=f_high, order=4)
v_high_f = filtering_zerophase(v_TSA_f, fs_re, 'band', f_low=f_low, f_high=f_high, order=4)

plt.figure(figsize=(9, 4))
plt.plot(v_high_f, color='C3', label='Fault')
plt.plot(v_high_n, color='C0', label='Normal')
plt.xlabel('Samples'); plt.ylabel('v'); plt.legend()
plt.title(f'Highpass-like filter output (Order-domain, time view)')
plt.tight_layout(); plt.show()

In [ ]:
env_f = hilbert_envelope(v_high_f)
env_n = hilbert_envelope(v_high_n)
v_res = hilbert_envelope(env_f - env_n)

plt.figure(figsize=(9, 4))
plt.plot(v_res, color='C3')
plt.xlabel('Samples'); plt.ylabel('residual')
plt.title('Residual Envelope — sharp peaks at fault location')
plt.tight_layout(); plt.show()

**관찰**: `envelope(fault) − envelope(normal)` 의 차이는 "결함만 더 강해진 성분"을 시간 영역에서 추출. 이에 다시 envelope 를 취하면 sharp peak (결함 impulse 위치) 가 남는다.

## 정리

- 회전 속도 변동 신호의 일반 FFT는 스펙트럼이 퍼져 해석이 어렵다.
- 각도 기반 재샘플링으로 시간축 → 각도축 변환하면 **Order 스펙트럼**에서 첨예한 피크를 얻는다.
- 정수 회전수를 맞추는 방법: **Zero-padding** 또는 **Trigger (trig_rot=1)**.
- **TSA**: 회전주기로 reshape + 평균 → 비동기 노이즈 제거, 주기적 이상치 강조.
- **하이패스 + 힐베르트 포락선**: 기본 성분을 제거하고 고장 임펄스 위치를 가시화.

### 생각해보기
1. `fs_re` (회전당 샘플 수) 를 50 / 200 으로 바꾸면 Order 분해능에 어떤 차이가 있는가?
2. 회전 속도 프로파일이 비선형(예: $1 + 0.1 t^2$) 이면 재샘플링 결과는 어떻게 달라질까?
3. TSA 세그먼트 개수(평균 횟수)를 늘리면 SNR 개선은 얼마나 되는가? (√N 법칙 확인)